# Investor Lens Analyzer — Phase 2: Company Analysis

This notebook applies an investor's DNA framework to a target company.

**To switch investor or company:** edit only the `⚙️ CONFIGURATION` cell below — nothing else needs changing.

## ⚙️ CONFIGURATION — Edit this cell to switch investor or company

In [22]:
# ============================================================
#  USER CONFIG — only edit this cell
# ============================================================

# -- Investor (must match a key in INVESTOR_REGISTRY) --------
ACTIVE_INVESTOR = "warren_buffett"   # ← CHANGE to switch investor

INVESTOR_REGISTRY = {
    "warren_buffett":  {"display_name": "Warren Buffett",  "style": "Quality Compounder"},
    "howard_marks":    {"display_name": "Howard Marks",    "style": "Credit/Cycles"},
    "seth_klarman":    {"display_name": "Seth Klarman",    "style": "Value/Distressed"},
    "peter_lynch":     {"display_name": "Peter Lynch",     "style": "Value/Growth"},
    "nick_sleep":      {"display_name": "Nick Sleep",      "style": "Value/Distressed"},
}

# -- Target Company ------------------------------------------
COMPANY_NAME    = "Apple Inc."          # ← CHANGE company name
TICKER          = "AAPL"                # ← CHANGE ticker
CURRENT_PRICE   = "$287.42"             # ← CHANGE to current price
MARKET_CAP      = "$4.22 trillion"       # ← CHANGE to current market cap

# Short description of the company / opportunity
TARGET_DESCRIPTION = """
Apple Inc. is a global technology company focused on consumer electronics, software, and services,
with core products including the iPhone, Mac, iPad, and wearables, alongside a rapidly growing
services ecosystem (App Store, iCloud, Apple Music, Apple Pay).

The company operates a highly integrated hardware-software ecosystem that drives strong customer
loyalty, recurring revenue, and pricing power. Apple generates substantial free cash flow supported
by premium branding, global distribution, and high-margin services, and returns capital through
share buybacks and dividends.
"""

# ============================================================
#  Derive variables automatically — do not edit below
# ============================================================
from datetime import date
cfg = INVESTOR_REGISTRY[ACTIVE_INVESTOR]
INVESTOR_NAME         = ACTIVE_INVESTOR
INVESTOR_DISPLAY_NAME = cfg["display_name"]
INVESTMENT_STYLE      = cfg["style"]
ANALYSIS_DATE         = str(date.today())

print(f"✅ Investor : {INVESTOR_DISPLAY_NAME} ({INVESTMENT_STYLE})")
print(f"   Company  : {COMPANY_NAME} ({TICKER}) @ {CURRENT_PRICE}")
print(f"   Date     : {ANALYSIS_DATE}")

✅ Investor : Warren Buffett (Quality Compounder)
   Company  : Apple Inc. (AAPL) @ $287.42
   Date     : 2026-05-08


## 1. Setup & Authentication

In [23]:
import json
from pathlib import Path

from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

import google.auth
from google.auth.transport.requests import Request
from google import genai
from google.genai import types

PROJECT_ID = "analytics-in-practice-492209"
LOCATION   = "us-central1"
MODEL_NAME = "gemini-2.5-pro"

SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=SCOPES)
creds.refresh(Request())

client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=LOCATION, credentials=creds
)
print("✅ Gemini client ready")

Mounted at /content/drive
✅ Gemini client ready


## 2. Load Phase 1 Outputs

In [24]:
from pathlib import Path
import json

BASE_DIR = Path("/content/drive/MyDrive/UC-A8 Investor Lens Analyzer")

DNA_DIR = BASE_DIR / "Investor DNA"

ANALYSIS_DIR = BASE_DIR / "Investor Analysis"

DNA_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

dna_path = DNA_DIR / f"{INVESTOR_NAME}_dna.md"
evidence_path = DNA_DIR / f"{INVESTOR_NAME}_evidence.json"

print("Looking for Phase 1 files:")
print("DNA path:", dna_path)
print("Evidence path:", evidence_path)

if not dna_path.exists():
    raise FileNotFoundError(
        f"DNA file not found: {dna_path}\n"
        "Please check that Phase 1 saved this investor's DNA file into the Investor DNA folder."
    )

if not evidence_path.exists():
    raise FileNotFoundError(
        f"Evidence file not found: {evidence_path}\n"
        "Please check that Phase 1 saved this investor's evidence JSON into the Investor DNA folder."
    )

final_dna_markdown = dna_path.read_text(encoding="utf-8")

with open(evidence_path, "r", encoding="utf-8") as f:
    merged_evidence = json.load(f)

print(f"✅ Loaded Phase 1 outputs for {INVESTOR_DISPLAY_NAME}")
print(f"   DNA chars     : {len(final_dna_markdown):,}")
print(f"   Evidence keys : {list(merged_evidence.keys())[:5]} ...")

Looking for Phase 1 files:
DNA path: /content/drive/MyDrive/UC-A8 Investor Lens Analyzer/Investor DNA/warren_buffett_dna.md
Evidence path: /content/drive/MyDrive/UC-A8 Investor Lens Analyzer/Investor DNA/warren_buffett_evidence.json
✅ Loaded Phase 1 outputs for Warren Buffett
   DNA chars     : 14,741
   Evidence keys : ['investment_philosophy', 'invariable_truths', 'buy_criteria', 'avoid_red_flags', 'position_sizing_portfolio'] ...


## 3. Build & Run Phase 2 Analysis

In [25]:
def build_phase2_prompt(
    investor_display_name, investment_style,
    company_name, ticker, current_price, market_cap,
    analysis_date, target_description,
    investor_dna, evidence
) -> str:
    evidence_text = json.dumps(evidence, indent=2, ensure_ascii=False)
    return f"""
You are applying {investor_display_name}'s investment framework to a specific company.

Your job is NOT to give a generic stock pitch.
Your job is to simulate how THIS investor would analyze the target,
based ONLY on the provided Investor DNA and structured evidence.

Rules:
- Use only the provided Investor DNA and structured evidence for investor philosophy.
- Do not invent investor quotes.
- If company-specific information is not provided, label it as an assumption or limitation.
- Make the analysis specific to the target company.
- Keep the tone professional and analytical — like an investment memo.
- Use only quotes from the provided evidence.

Use this exact output structure:

{investor_display_name.upper()} ANALYSIS: {company_name}

Lens Applied: {investor_display_name} | {investment_style}
Target: {company_name} ({ticker}) | Current Price: {current_price} | Market Cap: {market_cap}
Analysis Date: {analysis_date}

─────────────────────────────────────────────

INITIAL SCREEN
Would this pass {investor_display_name}'s initial filter?
[ ] YES - Fits core criteria, worth deep investigation
[ ] MAYBE - Some appeal but significant questions
[ ] NO - Doesn't match what they look for

Reasoning: [2-3 sentences]

─────────────────────────────────────────────

WHAT {investor_display_name.upper()} WOULD LIKE
• [Positive Factor 1]
  [Explanation]
  > "[Supporting quote from provided evidence]"

• [Positive Factor 2]
  [Explanation]
  > "[Supporting quote]"

• [Positive Factor 3]
  [Explanation]
  > "[Supporting quote]"

─────────────────────────────────────────────

WHAT {investor_display_name.upper()} WOULD WORRY ABOUT
• [Concern 1]
  [Explanation]
  > "[Supporting quote]"

• [Concern 2]
  [Explanation]
  > "[Supporting quote]"

• [Concern 3]
  [Explanation]
  > "[Supporting quote]"

─────────────────────────────────────────────

KEY QUESTIONS {investor_display_name.upper()} WOULD ASK
1. [Question reflecting their priorities]
2. [Question reflecting their concerns]
3. [Question reflecting their process]
4. [Question about risk/downside]
5. [Question about valuation/entry]

─────────────────────────────────────────────

HISTORICAL PARALLELS
Similar Investments They Made:
• [Company] ([Year]): [Brief description and relevance]
• [Company] ([Year]): [Brief description and relevance]

Similar Situations They Avoided:
• [Company/Situation]: [Why they passed]

─────────────────────────────────────────────

LIKELY VERDICT
{investor_display_name}'s Probable Conclusion:
[ ] STRONG BUY - High conviction, fits core criteria perfectly
[ ] INTERESTED - Worth building position, monitoring closely
[ ] WATCHLIST - Interesting but needs [specific condition]
[ ] PASS - Doesn't fit framework because [reason]
[ ] HARD PASS - Active red flags: [reason]

Position Sizing Estimate: [If bought, what size?]
Catalyst Dependency: [Need catalyst or buy and wait?]

─────────────────────────────────────────────

CONFIDENCE CALIBRATION
• Framework fit: [High/Medium/Low]
• Data quality: [High/Medium/Low]
• Historical precedent: [High/Medium/Low]

Caveats: [Limitations in applying this lens]

═══════════════════════════════════════════════
INVESTOR DNA (for reference):
{investor_dna}

STRUCTURED EVIDENCE:
{evidence_text}

COMPANY / OPPORTUNITY:
{target_description}
"""

phase2_prompt = build_phase2_prompt(
    investor_display_name=INVESTOR_DISPLAY_NAME,
    investment_style=INVESTMENT_STYLE,
    company_name=COMPANY_NAME,
    ticker=TICKER,
    current_price=CURRENT_PRICE,
    market_cap=MARKET_CAP,
    analysis_date=ANALYSIS_DATE,
    target_description=TARGET_DESCRIPTION,
    investor_dna=final_dna_markdown,
    evidence=merged_evidence
)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=phase2_prompt,
    config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=6000)
)

phase2_analysis = response.text
print(phase2_analysis)

WARREN BUFFETT ANALYSIS: Apple Inc.

Lens Applied: Warren Buffett | Quality Compounder
Target: Apple Inc. (AAPL) | Current Price: $287.42 | Market Cap: $4.22 trillion
Analysis Date: 2026-05-08

─────────────────────────────────────────────

INITIAL SCREEN
Would this pass Warren Buffett's initial filter?
[X] YES - Fits core criteria, worth deep investigation
[ ] MAYBE - Some appeal but significant questions
[ ] NO - Doesn't match what they look for

Reasoning: Apple is large enough to be a meaningful investment for Berkshire, satisfying the size criteria. Its business is centered on a powerful, understandable consumer brand and an ecosystem that creates a durable competitive advantage. The company's immense and consistent cash generation aligns perfectly with the focus on intrinsic value derived from cash that can be taken out of a business.

─────────────────────────────────────────────

WHAT WARREN BUFFETT WOULD LIKE
• **Durable Competitive Advantage (The Ecosystem Moat)**
  Apple's i

## 4. Save Analysis to Google Drive

In [26]:
phase2_path = ANALYSIS_DIR / f"{INVESTOR_NAME}_{TICKER.lower()}_phase2_analysis.md"
phase2_path.write_text(phase2_analysis, encoding="utf-8")

print(f"✅ Saved Phase 2 analysis to: {phase2_path}")
print(f"   Size: {phase2_path.stat().st_size:,} bytes")

✅ Saved Phase 2 analysis to: /content/drive/MyDrive/UC-A8 Investor Lens Analyzer/Investor Analysis/warren_buffett_aapl_phase2_analysis.md
   Size: 8,107 bytes
